# Moshi → Bridge → AvatarForcing Unified Streaming Pipeline

**Real-Time Speech-to-Avatar on RunPod GPU**

```
User Audio → Moshi LM (tokens @12.5Hz) → Bridge (wav2vec features @25Hz) → AvatarForcing (video @25fps)
```

In [ ]:
!git clone https://github.com/MoshiHead/Moshi-AvatarForcing-bridge-v2.git

Cloning into 'Moshi-AvatarForcing-bridge-v1'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 145 (delta 6), reused 142 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 1.10 MiB | 13.46 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [ ]:
%cd /workspace/Moshi-AvatarForcing-bridge-v2

/workspace/Moshi-AvatarForcing-bridge-v1


/usr/local/lib/python3.11/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
import torch, sys
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    total_gb = p.total_memory / 1_000_000_000
    print(f'  GPU {i}: {p.name} | {total_gb:.1f} GB')
print('Python:', sys.version.split()[0])


PyTorch: 2.4.1+cu124
CUDA: True
  GPU 0: NVIDIA A100-SXM4-80GB | 85.1 GB
Python: 3.11.10


In [4]:
import subprocess, sys
# System deps
subprocess.run('apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1'.split('&&')[0].split(), check=True, capture_output=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg','libsndfile1','git'], check=True, capture_output=True)
print('System deps OK')


System deps OK


In [5]:
import subprocess
import sys

pkgs = [
    "moshi==0.2.4",
    "sphn>=0.1.4",
    "sentencepiece>=0.1.99",
    "transformers==4.41.2",
    "accelerate==0.31.0",
    "diffusers==0.30.0",
    "peft==0.13.0",
    "ftfy",
    "easydict",
    "safetensors>=0.4.0",
    "huggingface_hub>=0.23.0",
    "omegaconf>=2.3.0",
    "einops>=0.7.0",
    "soundfile>=0.12.1",
    "pyyaml>=6.0",
    "pillow>=10.0",
    "decord>=0.6.0",
    "websockets>=12.0",
    "lmdb>=1.4.0",
    "pandas>=2.0",
    "opencv-python-headless>=4.8",
    "pyworld>=0.3.4",
    "tqdm>=4.66",
    "ipywidgets>=8.0",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
    check=True
)

print("Python packages installed")


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


Python packages installed


In [6]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124

Found existing installation: torch 2.4.1+cu124
Uninstalling torch-2.4.1+cu124:
  Successfully uninstalled torch-2.4.1+cu124
Found existing installation: torchvision 0.19.1+cu124
Uninstalling torchvision-0.19.1+cu124:
  Successfully uninstalled torchvision-0.19.1+cu124
Found existing installation: torchaudio 2.4.1+cu124
Uninstalling torchaudio-2.4.1+cu124:
  Successfully uninstalled torchaudio-2.4.1+cu124
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.3/908.3 MB 122.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 129.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 127.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 196.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 84.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 202.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
# Remove broken flash-attn to force PyTorch SDPA fallback (much more stable)

import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "flash-attn", "xformers"],
    check=False,
    capture_output=True
)

print("Removed flash-attn/xformers to use native PyTorch SDPA instead (avoids C++ ABI crashes)")

Removed flash-attn/xformers to use native PyTorch SDPA instead (avoids C++ ABI crashes)


In [ ]:
import sys
from pathlib import Path

# ─── CONFIGURE THESE ─────────────────────────────────────────────────────────
WORKSPACE    = Path('/workspace')
BRIDGE_ROOT  = WORKSPACE / 'Moshi-AvatarForcing-bridge-v2'
MOSHI_ROOT   = BRIDGE_ROOT / 'moshi-inference'
AF_ROOT      = BRIDGE_ROOT / 'AvatarForcing-inference'
BRIDGE_MOD   = BRIDGE_ROOT / 'bridge_module'
UNIFIED      = BRIDGE_ROOT / 'unified_pipeline'
WAN_DIR      = AF_ROOT / 'wan_models'
BRIDGE_CKPT  = BRIDGE_ROOT / 'checkpoints' / 'bridge_best.pt'
BRIDGE_CFG   = BRIDGE_MOD / 'config.yaml'
AF_CKPT      = AF_ROOT / 'checkpoints' / 'ode_audio_init.pt'
AF_CFG       = AF_ROOT / 'configs' / 'avatarforcing.yaml'
# ─────────────────────────────────────────────────────────────────────────────

for p in [str(BRIDGE_ROOT), str(MOSHI_ROOT), str(AF_ROOT), str(BRIDGE_MOD)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Paths set up:')
for p in [BRIDGE_ROOT, MOSHI_ROOT, AF_ROOT, BRIDGE_MOD]:
    print(' ', p, '✓' if p.exists() else '✗ MISSING')


Paths set up:
  /workspace/Moshi-AvatarForcing-bridge-v1 ✓
  /workspace/Moshi-AvatarForcing-bridge-v1/moshi-inference ✓
  /workspace/Moshi-AvatarForcing-bridge-v1/AvatarForcing-inference ✓
  /workspace/Moshi-AvatarForcing-bridge-v1/bridge_module ✓


In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

# Checkpoints folder
checkpoints_dir = Path("/workspace/Moshi-AvatarForcing-bridge-v1/checkpoints")

# Create directory if it does not exist
checkpoints_dir.mkdir(parents=True, exist_ok=True)

print("Downloading bridge_best.pt from dataset repo...")

snapshot_download(
    repo_id="Darknsu/bridge_module_wav2vec_librispeech_v1",
    repo_type="dataset",                 # dataset repo
    local_dir=str(checkpoints_dir),
    allow_patterns=["bridge_best.pt"],   # only download this file
    ignore_patterns=["*.md"],
)

print("bridge_best.pt downloaded successfully!")

In [ ]:
from huggingface_hub import snapshot_download

WAN_DIR.mkdir(parents=True, exist_ok=True)

print('Downloading Wan2.1-T2V-1.3B (large, ~14GB)...')
snapshot_download(
    repo_id='Wan-AI/Wan2.1-T2V-1.3B',
    local_dir=str(WAN_DIR / 'Wan2.1-T2V-1.3B'),
    ignore_patterns=['*.md'],
)
print('Wan2.1 OK')

print('Downloading wav2vec2-base-960h...')
snapshot_download(
    repo_id='facebook/wav2vec2-base-960h',
    local_dir=str(WAN_DIR / 'wav2vec2-base-960h'),
)
print('wav2vec2 OK')


In [11]:
from pathlib import Path
from huggingface_hub import snapshot_download

# Checkpoints folder
checkpoints_dir_af = Path(
    "/workspace/Moshi-AvatarForcing-bridge-v1/AvatarForcing-inference/checkpoints"
)

# Create directory if it does not exist
checkpoints_dir_af.mkdir(parents=True, exist_ok=True)

print("Downloading AvatarForcing model...")

snapshot_download(
    repo_id="lycui/AvatarForcing",
    local_dir=str(checkpoints_dir_af),
    ignore_patterns=["*.md"],
)

print("AvatarForcing downloaded successfully!")

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

ode_audio_init.pt:   0%|          | 0.00/6.39G [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

model.pt:   0%|          | 0.00/19.2G [00:00<?, ?B/s]

AvatarForcing downloaded successfully!


In [ ]:
import sys
from pathlib import Path

# ─── CONFIGURE THESE ─────────────────────────────────────────────────────────
WORKSPACE    = Path('/workspace')
BRIDGE_ROOT  = WORKSPACE / 'Moshi-AvatarForcing-bridge-v2'
MOSHI_ROOT   = BRIDGE_ROOT / 'moshi-inference'
AF_ROOT      = BRIDGE_ROOT / 'AvatarForcing-inference'
BRIDGE_MOD   = BRIDGE_ROOT / 'bridge_module'
UNIFIED      = BRIDGE_ROOT / 'unified_pipeline'
WAN_DIR      = AF_ROOT / 'wan_models'
BRIDGE_CKPT  = BRIDGE_ROOT / 'checkpoints' / 'bridge_best.pt'
BRIDGE_CFG   = BRIDGE_MOD / 'config.yaml'
AF_CKPT      = AF_ROOT / 'checkpoints' / 'ode_audio_init.pt'
AF_CFG       = AF_ROOT / 'configs' / 'avatarforcing.yaml'
# ─────────────────────────────────────────────────────────────────────────────

for p in [str(BRIDGE_ROOT), str(MOSHI_ROOT), str(AF_ROOT), str(BRIDGE_MOD)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Paths set up:')
for p in [BRIDGE_ROOT, MOSHI_ROOT, AF_ROOT, BRIDGE_MOD]:
    print(' ', p, '✓' if p.exists() else '✗ MISSING')


Paths set up:
  /workspace/Moshi-AvatarForcing-bridge-v1 ✓
  /workspace/Moshi-AvatarForcing-bridge-v1/moshi-inference ✓
  /workspace/Moshi-AvatarForcing-bridge-v1/AvatarForcing-inference ✓
  /workspace/Moshi-AvatarForcing-bridge-v1/bridge_module ✓


In [2]:
# Upload reference image and input audio
import ipywidgets as widgets
from IPython.display import display

upload_img = widgets.FileUpload(accept='image/*', multiple=False, description='Portrait Image')
upload_aud = widgets.FileUpload(accept='audio/*', multiple=False, description='Input Audio')
display(widgets.VBox([upload_img, upload_aud]))
print('Upload your files above, then run the next cell.')


Upload your files above, then run the next cell.


In [3]:
from pathlib import Path

UPLOADS = BRIDGE_ROOT / 'uploads'
UPLOADS.mkdir(exist_ok=True)

# Save image
if upload_img.value:
    fname, data = next(iter(upload_img.value.items()))
    IMAGE_PATH = str(UPLOADS / fname)
    Path(IMAGE_PATH).write_bytes(data['content'])
    print('Image saved:', IMAGE_PATH)
else:
    IMAGE_PATH = '/workspace/Moshi-AvatarForcing-bridge-v1/uploads/image.jpg'  # set manually if not using widget
    print('WARN: no upload — set IMAGE_PATH manually')

# Save audio
if upload_aud.value:
    fname, data = next(iter(upload_aud.value.items()))
    AUDIO_PATH = str(UPLOADS / fname)
    Path(AUDIO_PATH).write_bytes(data['content'])
    print('Audio saved:', AUDIO_PATH)
else:
    AUDIO_PATH = '/workspace/Moshi-AvatarForcing-bridge-v1/uploads/wav.mp3'  # set manually
    print('WARN: no upload — set AUDIO_PATH manually')

OUTPUT_PATH = str(BRIDGE_ROOT / 'output' / 'avatar_output.mp4')
Path(OUTPUT_PATH).parent.mkdir(exist_ok=True)
print('Output path:', OUTPUT_PATH)


WARN: no upload — set IMAGE_PATH manually
WARN: no upload — set AUDIO_PATH manually
Output path: /workspace/Moshi-AvatarForcing-bridge-v1/output/avatar_output.mp4


In [4]:
import logging, torch
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s - %(message)s')

from unified_pipeline.config import (
    PipelineConfig, MoshiConfig, BridgeConfig, AvatarForcingConfig,
    DEFAULT_AVATAR_PROMPT
)

cfg = PipelineConfig(
    moshi=MoshiConfig(
        hf_repo='kyutai/moshiko-pytorch-bf16',
        device='cuda',
        dtype=torch.bfloat16,
        use_sampling=True,
        temp=0.8, temp_text=0.7, top_k=250,
    ),
    bridge=BridgeConfig(
        checkpoint_path=str(BRIDGE_CKPT),
        config_path=str(BRIDGE_CFG),
        device='cuda',
        dtype=torch.bfloat16,
        chunk_tokens=4,      # 4 moshi frames = 320ms per bridge chunk
        upsample_factor=2,   # 12.5Hz tokens x2 = 25Hz audio features
        use_kv_cache=True,
        use_projection=True, # project bridge 768-dim -> 9984-dim
        wav2vec_num_layers=13,
    ),
    avatar=AvatarForcingConfig(
        avatarforcing_root=str(AF_ROOT),
        config_path=str(AF_CFG),
        checkpoint_path=str(AF_CKPT),
        device='cuda',
        dtype=torch.bfloat16,
        num_output_frames=21,
        fps=25,
        prompt=DEFAULT_AVATAR_PROMPT,
        seed=42,
    ),
    token_queue_maxsize=128,
    feature_queue_maxsize=64,
    frame_queue_maxsize=32,
)

print('Config OK')
print('Prompt:', cfg.avatar.prompt[:100], '...')


Config OK
Prompt: A realistic person speaking naturally with accurate lip synchronization, expressive facial motion, s ...


In [5]:
import gc, torch
torch.cuda.empty_cache(); gc.collect()

from unified_pipeline.model_loader import load_all_models

print('Loading all models (first run downloads ~20GB, may take 10-15 min)...')
models = load_all_models(cfg)

mimi       = models['mimi']
lm_gen     = models['lm_gen']
bridge     = models['bridge']
projection = models['projection']
av_pipe    = models['pipeline']

alloc = torch.cuda.memory_allocated() / 1e9
res   = torch.cuda.memory_reserved()  / 1e9
print(f'\nAll models loaded!')
print(f'VRAM: {alloc:.2f} GB allocated / {res:.2f} GB reserved')


2026-05-09 03:45:00,743 [INFO] unified_pipeline.model_loader - ============================================================
2026-05-09 03:45:00,744 [INFO] unified_pipeline.model_loader - Loading Moshi…


Loading all models (first run downloads ~20GB, may take 10-15 min)...


2026-05-09 03:45:01,171 [INFO] unified_pipeline.model_loader - Loading Moshi checkpoint from: kyutai/moshiko-pytorch-bf16
/workspace/Moshi-AvatarForcing-bridge-v1/moshi-inference/moshi/models/loaders.py:204: UserWarning: Repository kyutai/moshiko-pytorch-bf16 contains no config.json. Assuming this is a Moshi 7B. Support for such repository might be removed in the future.
  warnings.warn(
2026-05-09 03:45:01,362 [INFO] unified_pipeline.model_loader - Loading Mimi audio codec…
2026-05-09 03:45:02,512 [INFO] unified_pipeline.model_loader - [VRAM] after Mimi → allocated=0.39 GB | reserved=0.80 GB
2026-05-09 03:45:02,545 [INFO] unified_pipeline.model_loader - Loading Moshi language model…
2026-05-09 03:45:06,311 [INFO] unified_pipeline.model_loader - [VRAM] after Moshi LM → allocated=15.77 GB | reserved=16.02 GB
2026-05-09 03:45:06,334 [INFO] unified_pipeline.model_loader - Moshi loaded | frame_size=1920 | sample_rate=24000 | frame_rate=12.5
2026-05-09 03:45:06,446 [INFO] unified_pipeline.m

KV inference with 4 frames per block
Use LoRA: rank=128, alpha=64.0


2026-05-09 03:47:21,879 [INFO] unified_pipeline.model_loader - AvatarForcing checkpoint loaded
2026-05-09 03:47:50,845 [INFO] unified_pipeline.model_loader - WAV2VEC bypassed — bridge features will be injected directly
2026-05-09 03:47:50,847 [INFO] unified_pipeline.model_loader - [VRAM] after AvatarForcing → allocated=32.23 GB | reserved=32.66 GB
2026-05-09 03:47:50,848 [INFO] unified_pipeline.model_loader - AvatarForcing loaded | block_size=4 | denoise_steps=[1000.0, 952.3810424804688, 769.2307739257812, 555.5556030273438]
2026-05-09 03:47:51,956 [INFO] unified_pipeline.model_loader - [VRAM] ALL MODELS LOADED → allocated=32.23 GB | reserved=32.66 GB
2026-05-09 03:47:51,957 [INFO] unified_pipeline.model_loader - ============================================================
2026-05-09 03:47:51,958 [INFO] unified_pipeline.model_loader - All models loaded successfully



All models loaded!
VRAM: 32.23 GB allocated / 32.66 GB reserved


In [6]:
from unified_pipeline.async_pipeline import UnifiedPipeline

unified = UnifiedPipeline(cfg, models)
unified.setup_reference_image(IMAGE_PATH)
print('Pipeline ready - reference image encoded')


2026-05-09 03:47:58,812 [INFO] unified_pipeline.avatarforcing_streamer - StreamingAvatarGenerator initialized | block_size=4 frames
2026-05-09 03:47:58,813 [INFO] unified_pipeline.async_pipeline - UnifiedPipeline ready | audio_dim=10752 | chunk_tokens=4 | block_size=4
2026-05-09 03:47:58,868 [INFO] unified_pipeline.avatarforcing_streamer - Reference image loaded: /workspace/Moshi-AvatarForcing-bridge-v1/uploads/image.jpg → tensor [1, 3, 1, 544, 704]
2026-05-09 03:47:58,869 [INFO] unified_pipeline.avatarforcing_streamer - Encoding reference image…
2026-05-09 03:47:59,111 [INFO] unified_pipeline.avatarforcing_streamer - Reference image encoded → latent [1, 1, 16, 68, 88]
2026-05-09 03:47:59,113 [INFO] unified_pipeline.async_pipeline - Reference image ready: /workspace/Moshi-AvatarForcing-bridge-v1/uploads/image.jpg


Pipeline ready - reference image encoded


In [7]:
import sphn, torch

print('Loading audio:', AUDIO_PATH)
in_pcms, _ = sphn.read(AUDIO_PATH, sample_rate=mimi.sample_rate)
audio_tensor = torch.from_numpy(in_pcms).cuda()[None, 0:1]  # [1,1,N]
duration_s = audio_tensor.shape[-1] / mimi.sample_rate
n_samples  = audio_tensor.shape[-1]
print(f'Audio: {duration_s:.2f}s | {n_samples} samples @ {mimi.sample_rate}Hz')


Loading audio: /workspace/Moshi-AvatarForcing-bridge-v1/uploads/wav.mp3
Audio: 49.65s | 1191680 samples @ 24000Hz


In [8]:
import asyncio, time

print('Starting streaming pipeline...')
print(f'Output: {OUTPUT_PATH}')

t0 = time.monotonic()

async def run():
    return await unified.run_to_video(
        audio_tensor,
        output_path=OUTPUT_PATH,
        fps=cfg.avatar.fps,
        max_blocks=64,
    )

# Run async pipeline
try:
    loop = asyncio.get_event_loop()
    if loop.is_running():
        import nest_asyncio; nest_asyncio.apply()
        out_path = loop.run_until_complete(run())
    else:
        out_path = asyncio.run(run())
except Exception:
    import nest_asyncio; nest_asyncio.apply()
    out_path = asyncio.get_event_loop().run_until_complete(run())

elapsed = time.monotonic() - t0
n_frames = unified.avatar_generator._generated_frames
fps_achieved = n_frames / max(elapsed, 0.01)
print(f'Done in {elapsed:.1f}s | {n_frames} frames | {fps_achieved:.1f} fps achieved')
print('Output:', out_path)


2026-05-09 03:48:02,563 [INFO] unified_pipeline.async_pipeline - Pipeline run started | audio_samples=1191680 | max_blocks=64


Starting streaming pipeline...
Output: /workspace/Moshi-AvatarForcing-bridge-v1/output/avatar_output.mp4


2026-05-09 03:48:09,108 [INFO] unified_pipeline.avatarforcing_streamer - Encoding text prompt…
2026-05-09 03:48:09,197 [INFO] unified_pipeline.avatarforcing_streamer - Text prompt encoded
2026-05-09 03:48:09,789 [INFO] unified_pipeline.avatarforcing_streamer - KV cache prefilled with reference image frame
2026-05-09 03:49:41,267 [INFO] unified_pipeline.avatarforcing_streamer - StreamingAvatarGenerator: generation complete
2026-05-09 03:49:41,273 [INFO] unified_pipeline.async_pipeline - Avatar task complete
2026-05-09 03:49:41,300 [INFO] unified_pipeline.async_pipeline - Pipeline complete | frames=832 | total_time=98.74s | fps=8.4
2026-05-09 03:49:52,674 [INFO] unified_pipeline.async_pipeline - Video saved: /workspace/Moshi-AvatarForcing-bridge-v1/output/avatar_output.mp4 | 832 frames @ 25 fps


Done in 110.1s | 832 frames | 7.6 fps achieved
Output: /workspace/Moshi-AvatarForcing-bridge-v1/output/avatar_output.mp4


In [14]:
import asyncio, time
# OUTPUT_PATH = '/workspace/Moshi-AvatarForcing-bridge-v1/output/output.mp4'
print('Starting streaming pipeline...')
print(f'Output: {OUTPUT_PATH}')

t0 = time.monotonic()

# Run async pipeline natively in Jupyter
out_path = await unified.run_to_video(
    audio_tensor,
    output_path=OUTPUT_PATH,
    fps=cfg.avatar.fps,
    max_blocks=64,
)

elapsed = time.monotonic() - t0
n_frames = unified.avatar_generator._generated_frames
fps_achieved = n_frames / max(elapsed, 0.01)
print(f'Done in {elapsed:.1f}s | {n_frames} frames | {fps_achieved:.1f} fps achieved')
print('Output:', out_path)


2026-05-09 03:27:32,389 [INFO] unified_pipeline.async_pipeline - Pipeline run started | audio_samples=1191680 | max_blocks=64


Starting streaming pipeline...
Output: /workspace/Moshi-AvatarForcing-bridge-v1/output/avatar_output.mp4


2026-05-09 03:27:37,809 [INFO] unified_pipeline.avatarforcing_streamer - Encoding text prompt…
2026-05-09 03:27:37,927 [INFO] unified_pipeline.avatarforcing_streamer - Text prompt encoded
2026-05-09 03:27:38,551 [INFO] unified_pipeline.avatarforcing_streamer - KV cache prefilled with reference image frame
2026-05-09 03:29:10,159 [INFO] unified_pipeline.avatarforcing_streamer - StreamingAvatarGenerator: generation complete
2026-05-09 03:29:10,161 [INFO] unified_pipeline.async_pipeline - Avatar task complete
2026-05-09 03:29:10,188 [INFO] unified_pipeline.async_pipeline - Pipeline complete | frames=832 | total_time=97.80s | fps=8.5
2026-05-09 03:29:17,627 [INFO] unified_pipeline.async_pipeline - Video saved: /workspace/Moshi-AvatarForcing-bridge-v1/output/avatar_output.mp4 | 832 frames @ 25 fps


Done in 105.2s | 832 frames | 7.9 fps achieved
Output: /workspace/Moshi-AvatarForcing-bridge-v1/output/avatar_output.mp4


In [ ]:
from IPython.display import Video, display
display(Video(OUTPUT_PATH, embed=True, width=640))


In [ ]:
# Streaming frame-by-frame preview (first 5 frames)
import asyncio
from PIL import Image as PILImage
from IPython.display import display as ipy_display
import io

print('Streaming preview (first 5 frames):')

async def preview_frames():
    count = 0
    async for frame in unified.run(audio_tensor, max_blocks=4):
        if count >= 5: break
        pil = PILImage.fromarray(frame.pixels.numpy())
        print(f'Frame {frame.frame_idx} | latency={frame.latency_ms:.0f}ms')
        ipy_display(pil)
        count += 1

loop = asyncio.get_event_loop()
import nest_asyncio; nest_asyncio.apply()
loop.run_until_complete(preview_frames())


In [ ]:
# Latency benchmark
import asyncio, nest_asyncio
nest_asyncio.apply()

print('Running 5s latency benchmark...')

async def bench():
    return await unified.benchmark_latency(audio_seconds=5.0)

results = asyncio.get_event_loop().run_until_complete(bench())
print('Latency results:')
for k, v in results.items():
    print(f'  {k}: {v}')


In [ ]:
# Optional: WebSocket server for browser client
# Uncomment and run to start real-time streaming server
# Expose port 8765 in RunPod pod settings

# import asyncio, nest_asyncio
# nest_asyncio.apply()
# from unified_pipeline.async_pipeline import StreamingPipelineServer
# server = StreamingPipelineServer(unified)
# print('Starting WebSocket server on :8765')
# asyncio.get_event_loop().run_until_complete(server.serve('0.0.0.0', 8765))

print('WebSocket server: uncomment above to enable')


In [ ]:
# Cleanup GPU memory
import gc, torch
del models, unified
gc.collect()
torch.cuda.empty_cache()
alloc = torch.cuda.memory_allocated() / 1e9
print(f'VRAM after cleanup: {alloc:.2f} GB allocated')
